# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BarnaliBhowmik/FlyRank_ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
!pip -q install duckdb huggingface_hub pyarrow

In [9]:
import duckdb
import pyarrow.dataset as ds
from huggingface_hub import HfFileSystem
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

fs = HfFileSystem(token=HF_TOKEN)

print("Hugging Face login successful.")

Hugging Face login successful.


In [10]:
march_path = (
    "datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/"
    "month=2026-03/"
)

# Get the March parquet files
march_files = fs.glob(march_path + "*.parquet")

print("March files found:", len(march_files))
print(march_files)

March files found: 1
['datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet']


In [11]:
march_file = march_files[0]

march_uri = "hf://" + march_file

march_dataset = ds.dataset(
    march_uri,
    format="parquet",
    filesystem=fs
)

print("March 2026 PyArrow dataset created.")

March 2026 PyArrow dataset created.


In [12]:
con = duckdb.connect()

con.register("march_data", march_dataset)

print("March 2026 data connected successfully.")

March 2026 data connected successfully.


In [13]:
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM march_data
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────┐
│ total_rows │ first_date │ last_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My response:
### Signal checks

Before creating the baseline rule, I will check two signals that support the
CTR / engagement opportunity idea.

**Signal 1 — CTR vs search position**

This checks whether observed CTR differs across search-position buckets.
Search position is relevant because FlyRank's CTR-fix logic also considers
CTR relative to position.

**Signal 2 — Search impressions**

This checks whether pages with larger search volume show a meaningful difference
in the potential opportunity. Impressions are relevant because a change on a
higher-volume page can affect more observed searches.

For each signal, I will report the number of rows (`n`) in each bucket and give
one verdict:

- CONFIRMED
- OPPOSITE
- MIXED
- FALSE

The verdict will be based on the observed March 2026 data rather than chosen
in advance.

### Planned baseline rule

After checking the two signals, I will encode one simple ranking rule using
only signals that are supported by the data and are available at the decision
moment.

The rule will produce:

1. a score,
2. one reason code, and
3. an action label.

The rule is a decision-support baseline, not a claim of causality.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [15]:
signal_1 = con.sql("""
    SELECT
        CASE
            WHEN gsc_avg_position >= 1 AND gsc_avg_position <= 3
                THEN '01-03'
            WHEN gsc_avg_position > 3 AND gsc_avg_position <= 5
                THEN '04-05'
            WHEN gsc_avg_position > 5 AND gsc_avg_position <= 10
                THEN '06-10'
            WHEN gsc_avg_position > 10 AND gsc_avg_position <= 20
                THEN '11-20'
            ELSE 'outside_1_20'
        END AS position_bucket,

        COUNT(*) AS n,

        SUM(gsc_clicks) AS clicks,
        SUM(gsc_impressions) AS impressions,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS observed_ctr

    FROM march_data

    WHERE gsc_data_available IS TRUE
      AND gsc_avg_position > 0

    GROUP BY 1
    ORDER BY 1
""")

signal_1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────┬────────┬─────────────┬───────────────────────┐
│ position_bucket │    n    │ clicks │ impressions │     observed_ctr      │
│     varchar     │  int64  │ int128 │   int128    │        double         │
├─────────────────┼─────────┼────────┼─────────────┼───────────────────────┤
│ 01-03           │  462625 │ 192659 │    46947845 │  0.004103681436283178 │
│ 04-05           │  535763 │ 241009 │    67873849 │  0.003550837377735864 │
│ 06-10           │  920359 │ 204819 │    69956264 │  0.002927815013105903 │
│ 11-20           │  519223 │  92449 │    29386006 │ 0.0031460212728466742 │
│ outside_1_20    │ 1009902 │  89722 │    66024879 │ 0.0013589119943710915 │
└─────────────────┴─────────┴────────┴─────────────┴───────────────────────┘

In [16]:
signal_2 = con.sql("""
    SELECT
        CASE
            WHEN gsc_impressions < 100
                THEN '<100'
            WHEN gsc_impressions < 500
                THEN '100-499'
            WHEN gsc_impressions < 1000
                THEN '500-999'
            WHEN gsc_impressions < 5000
                THEN '1K-4.9K'
            ELSE '5K+'
        END AS impressions_bucket,

        COUNT(*) AS n,

        AVG(gsc_avg_position) AS avg_position,

        SUM(gsc_clicks) AS clicks,
        SUM(gsc_impressions) AS impressions,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
            ELSE NULL
        END AS observed_ctr

    FROM march_data

    WHERE gsc_data_available IS TRUE
      AND gsc_impressions >= 0

    GROUP BY 1
    ORDER BY
        CASE impressions_bucket
            WHEN '<100' THEN 1
            WHEN '100-499' THEN 2
            WHEN '500-999' THEN 3
            WHEN '1K-4.9K' THEN 4
            WHEN '5K+' THEN 5
        END
""")

signal_2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬─────────┬────────────────────┬────────┬─────────────┬───────────────────────┐
│ impressions_bucket │    n    │    avg_position    │ clicks │ impressions │     observed_ctr      │
│      varchar       │  int64  │       double       │ int128 │   int128    │        double         │
├────────────────────┼─────────┼────────────────────┼────────┼─────────────┼───────────────────────┤
│ <100               │ 2972453 │  16.85041364392293 │ 172298 │    60168506 │  0.002863591128554862 │
│ 100-499            │  537157 │  10.94291530057118 │ 351515 │   114504214 │  0.003069887017433262 │
│ 500-999            │   69032 │ 11.594411229928776 │ 133694 │    47105216 │ 0.0028381994894153546 │
│ 1K-4.9K            │   31676 │  11.98946462439195 │ 142674 │    52577062 │ 0.0027136168240058755 │
│ 5K+                │     743 │  7.689288258115668 │  21651 │     6302591 │  0.003435253850360907 │
└────────────────────┴─────────┴────────────────────┴────────┴─────────────┴───────────────

### Signal verdicts

**Signal 1 — CTR vs search position: CONFIRMED**

Observed CTR is higher for pages ranking in positions 1–10 than for pages
outside the top 20. The relationship is not perfectly monotonic because the
11–20 bucket is slightly above the 06–10 bucket, but position is clearly
associated with observed CTR. This supports using search position in the
baseline rule.

**Signal 2 — Search impressions: MIXED**

Observed CTR does not increase consistently with impression volume. The
5K+ bucket has the highest observed CTR, but it contains only 743 rows,
while the lower-volume buckets contain substantially more rows. Therefore,
impression volume is not treated as a direct CTR signal. It may still be
used as a prioritization factor because higher-volume pages affect more
search impressions.

### Baseline rule decision

The baseline will therefore focus on **CTR relative to search position**,
with impressions used only to prioritize the size of the potential
opportunity.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## My response:

The baseline ranks pages that have lower observed CTR than the typical CTR
for their search-position bucket.

The score combines:

1. CTR gap relative to the position bucket benchmark.
2. Impression volume as a prioritization factor.

A larger positive CTR gap means the page has more potential CTR opportunity
relative to similar search positions. Higher impressions increase priority
because the opportunity affects more observed searches.

Only March 2026 observed fields are used. No future-window information,
label-derived fields, or FlyRank product flags are used.

Reason code:
- `LOW_CTR_FOR_POSITION` — observed CTR is below the typical CTR for its
  position bucket.

Action:
- `REVIEW_CTR` — send the page for human review.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [18]:
baseline_queue = con.sql("""
    WITH page_level AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_clicks) AS clicks,
            SUM(gsc_impressions) AS impressions,

            CASE
                WHEN SUM(gsc_impressions) > 0
                THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
                ELSE NULL
            END AS observed_ctr,

            AVG(gsc_avg_position) AS avg_position

        FROM march_data

        WHERE gsc_data_available IS TRUE
          AND gsc_avg_position > 0
          AND gsc_avg_position <= 20

        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    position_benchmarks AS (
        SELECT
            CASE
                WHEN avg_position <= 3 THEN '01-03'
                WHEN avg_position <= 5 THEN '04-05'
                WHEN avg_position <= 10 THEN '06-10'
                ELSE '11-20'
            END AS position_bucket,

            SUM(clicks) * 1.0 / SUM(impressions) AS benchmark_ctr

        FROM page_level

        WHERE impressions > 0

        GROUP BY 1
    ),

    scored AS (
        SELECT
            p.client_hash_id,
            p.content_hash_id,
            p.avg_position,
            p.impressions,
            p.clicks,
            p.observed_ctr,

            CASE
                WHEN p.avg_position <= 3 THEN '01-03'
                WHEN p.avg_position <= 5 THEN '04-05'
                WHEN p.avg_position <= 10 THEN '06-10'
                ELSE '11-20'
            END AS position_bucket,

            b.benchmark_ctr,

            GREATEST(b.benchmark_ctr - p.observed_ctr, 0) AS ctr_gap

        FROM page_level p

        JOIN position_benchmarks b
          ON (
                CASE
                    WHEN p.avg_position <= 3 THEN '01-03'
                    WHEN p.avg_position <= 5 THEN '04-05'
                    WHEN p.avg_position <= 10 THEN '06-10'
                    ELSE '11-20'
                END
             ) = b.position_bucket

        WHERE p.impressions > 0
    )

    SELECT
        ROW_NUMBER() OVER (
            ORDER BY
                ctr_gap * LN(1 + impressions) DESC,
                impressions DESC
        ) AS rank,

        client_hash_id,
        content_hash_id,

        ROUND(avg_position, 2) AS avg_position,
        position_bucket,

        impressions,
        clicks,

        ROUND(observed_ctr, 6) AS observed_ctr,
        ROUND(benchmark_ctr, 6) AS benchmark_ctr,
        ROUND(ctr_gap, 6) AS ctr_gap,

        ROUND(
            ctr_gap * LN(1 + impressions),
            4
        ) AS score,

        CASE
            WHEN ctr_gap > 0
                THEN 'LOW_CTR_FOR_POSITION'
            ELSE 'NO_CTR_GAP'
        END AS reason_code,

        CASE
            WHEN ctr_gap > 0
                THEN 'REVIEW_CTR'
            ELSE 'NO_ACTION'
        END AS action

    FROM scored

    ORDER BY rank
""")

baseline_queue

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────┬─────────────────────────┬──────────────────────────┬──────────────┬─────────────────┬─────────────┬────────┬──────────────┬───────────────┬──────────┬────────┬──────────────────────┬────────────┐
│ rank  │     client_hash_id      │     content_hash_id      │ avg_position │ position_bucket │ impressions │ clicks │ observed_ctr │ benchmark_ctr │ ctr_gap  │ score  │     reason_code      │   action   │
│ int64 │         varchar         │         varchar          │    double    │     varchar     │   int128    │ int128 │    double    │    double     │  double  │ double │       varchar        │  varchar   │
├───────┼─────────────────────────┼──────────────────────────┼──────────────┼─────────────────┼─────────────┼────────┼──────────────┼───────────────┼──────────┼────────┼──────────────────────┼────────────┤
│     1 │ client_1a730cb2640a1abf │ content_d61fc394d10cba41 │         2.74 │ 01-03           │       38000 │      1 │      2.6e-05 │      0.004045 │ 0.004019 │ 0.0424 │ LOW_CT

In [19]:
import os

os.makedirs("work/outputs", exist_ok=True)

baseline_queue.write_csv(
    "work/outputs/baseline_action_score.csv"
)

print("CSV written successfully:")
print("work/outputs/baseline_action_score.csv")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CSV written successfully:
work/outputs/baseline_action_score.csv


In [20]:
con.sql("""
    SELECT *
    FROM read_csv_auto(
        'work/outputs/baseline_action_score.csv'
    )
    LIMIT 10
""")

┌───────┬─────────────────────────┬──────────────────────────┬──────────────┬─────────────────┬─────────────┬────────┬──────────────┬───────────────┬──────────┬────────┬──────────────────────┬────────────┐
│ rank  │     client_hash_id      │     content_hash_id      │ avg_position │ position_bucket │ impressions │ clicks │ observed_ctr │ benchmark_ctr │ ctr_gap  │ score  │     reason_code      │   action   │
│ int64 │         varchar         │         varchar          │    double    │     varchar     │    int64    │ int64  │    double    │    double     │  double  │ double │       varchar        │  varchar   │
├───────┼─────────────────────────┼──────────────────────────┼──────────────┼─────────────────┼─────────────┼────────┼──────────────┼───────────────┼──────────┼────────┼──────────────────────┼────────────┤
│     1 │ client_1a730cb2640a1abf │ content_d61fc394d10cba41 │         2.74 │ 01-03           │       38000 │      1 │      2.6e-05 │      0.004045 │ 0.004019 │ 0.0424 │ LOW_CT

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## My response:

The top 20 pages are reviewed individually to check whether the baseline
ranking produces plausible review candidates.

For each page I record:

- **Action** — what the baseline recommends.
- **Reason code** — why the page was ranked.
- **Confidence note** — how strongly the available evidence supports the
  recommendation.
- **What would make it wrong** — a plausible reason the ranking could be
  misleading.

This is a skeptical human review. A high score does not prove that a page
has a genuine CTR improvement opportunity.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [22]:
top20 = con.sql("""
    SELECT
        rank,
        client_hash_id,
        content_hash_id,
        ROUND(avg_position, 2) AS avg_position,
        position_bucket,
        impressions,
        clicks,
        observed_ctr,
        benchmark_ctr,
        ctr_gap,
        score,
        reason_code,
        action
    FROM read_csv_auto(
        'work/outputs/baseline_action_score.csv'
    )
    WHERE rank <= 20
    ORDER BY rank
""")

top20

┌───────┬─────────────────────────┬──────────────────────────┬──────────────┬─────────────────┬─────────────┬────────┬──────────────┬───────────────┬──────────┬────────┬──────────────────────┬────────────┐
│ rank  │     client_hash_id      │     content_hash_id      │ avg_position │ position_bucket │ impressions │ clicks │ observed_ctr │ benchmark_ctr │ ctr_gap  │ score  │     reason_code      │   action   │
│ int64 │         varchar         │         varchar          │    double    │     varchar     │    int64    │ int64  │    double    │    double     │  double  │ double │       varchar        │  varchar   │
├───────┼─────────────────────────┼──────────────────────────┼──────────────┼─────────────────┼─────────────┼────────┼──────────────┼───────────────┼──────────┼────────┼──────────────────────┼────────────┤
│     1 │ client_1a730cb2640a1abf │ content_d61fc394d10cba41 │         2.74 │ 01-03           │       38000 │      1 │      2.6e-05 │      0.004045 │ 0.004019 │ 0.0424 │ LOW_CT

In [23]:
top20_review = con.sql("""
    SELECT
        rank,
        client_hash_id,
        content_hash_id,

        action,
        reason_code,

        CASE
            WHEN impressions >= 5000
                 AND ctr_gap >= 0.002
                THEN 'HIGH — large impression volume and clear CTR gap'

            WHEN impressions >= 500
                 AND ctr_gap >= 0.001
                THEN 'MEDIUM — meaningful volume and measurable CTR gap'

            ELSE
                'LOW — ranking signal is weaker or volume is limited'
        END AS confidence_note,

        CASE
            WHEN impressions < 500
                THEN 'Could be wrong because the page has limited impression volume.'

            WHEN avg_position > 10
                THEN 'Could be wrong because lower search position may itself explain the low CTR.'

            WHEN ctr_gap < 0.001
                THEN 'Could be wrong because the CTR gap is small.'

            ELSE
                'Could be wrong if the position-based CTR benchmark does not represent this page well.'
        END AS what_would_make_it_wrong

    FROM read_csv_auto(
        'work/outputs/baseline_action_score.csv'
    )

    WHERE rank <= 20

    ORDER BY rank
""")

top20_review

┌───────┬─────────────────────────┬──────────────────────────┬────────────┬──────────────────────┬──────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────┐
│ rank  │     client_hash_id      │     content_hash_id      │   action   │     reason_code      │                 confidence_note                  │                               what_would_make_it_wrong                                │
│ int64 │         varchar         │         varchar          │  varchar   │       varchar        │                     varchar                      │                                        varchar                                        │
├───────┼─────────────────────────┼──────────────────────────┼────────────┼──────────────────────┼──────────────────────────────────────────────────┼───────────────────────────────────────────────────────────────────────────────────────┤
│     1 │ client_1a730cb2640a1abf │ content_d61f

### Review summary

The baseline produces a ranked queue rather than a claim that every top-ranked
page is genuinely actionable.

The strongest candidates are those where the CTR gap is substantial and the
page has enough impressions for the opportunity to matter.

Potentially weak picks include pages with low impression volume, small CTR gaps,
or lower search positions where low CTR may simply reflect weaker visibility.

The top-20 review is therefore used as a sanity check before treating the
baseline as the benchmark for the Week-5 ML model.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## My response:

### Weak picks

A high baseline score does not automatically mean that a page is a good
human-review candidate.

I will flag top-ranked rows as potentially weak when:

- impression volume is low,
- the CTR gap is small, or
- the page is already in a lower search-position bucket where low CTR may
  simply reflect weaker visibility.

These are not removed automatically. They are recorded as cases where the
baseline may be less reliable.

### Leakage check

The baseline must use only information available from the March 2026
development data.

I will check that:

1. the source data is March 2026 only;
2. no future-window fields are used;
3. no FlyRank product flags are used as scoring inputs;
4. no label-derived fields such as `trend_pct` or `trend_direction` are used;
5. IDs are used only for grouping/output, not as predictive features.

The score, reason code, and action are outputs of the baseline, not input
features.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [25]:
weak_picks = con.sql("""
    SELECT
        rank,
        client_hash_id,
        content_hash_id,
        ROUND(avg_position, 2) AS avg_position,
        impressions,
        ROUND(observed_ctr, 6) AS observed_ctr,
        ROUND(benchmark_ctr, 6) AS benchmark_ctr,
        ROUND(ctr_gap, 6) AS ctr_gap,
        ROUND(score, 4) AS score,
        reason_code,
        action,
        CASE
            WHEN impressions < 500
                THEN 'Low impression volume'
            WHEN ctr_gap < 0.001
                THEN 'Small CTR gap'
            WHEN avg_position > 10
                THEN 'Lower search position may explain low CTR'
            ELSE
                'No obvious weakness'
        END AS weakness_note
    FROM read_csv_auto(
        'work/outputs/baseline_action_score.csv'
    )
    WHERE rank <= 20
    ORDER BY rank
""")

weak_picks

┌───────┬─────────────────────────┬──────────────────────────┬──────────────┬─────────────┬──────────────┬───────────────┬──────────┬────────┬──────────────────────┬────────────┬─────────────────────┐
│ rank  │     client_hash_id      │     content_hash_id      │ avg_position │ impressions │ observed_ctr │ benchmark_ctr │ ctr_gap  │ score  │     reason_code      │   action   │    weakness_note    │
│ int64 │         varchar         │         varchar          │    double    │    int64    │    double    │    double     │  double  │ double │       varchar        │  varchar   │       varchar       │
├───────┼─────────────────────────┼──────────────────────────┼──────────────┼─────────────┼──────────────┼───────────────┼──────────┼────────┼──────────────────────┼────────────┼─────────────────────┤
│     1 │ client_1a730cb2640a1abf │ content_d61fc394d10cba41 │         2.74 │       38000 │      2.6e-05 │      0.004045 │ 0.004019 │ 0.0424 │ LOW_CTR_FOR_POSITION │ REVIEW_CTR │ No obvious weakne

In [26]:
leakage_check = con.sql("""
    SELECT
        COUNT(*) AS march_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM march_data
""")

leakage_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────┬────────────┐
│ march_rows │ first_date │ last_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

In [27]:
print("LEAKAGE CHECK")
print("Source window: March 2026 only")
print("Future-window fields used: NO")
print("FlyRank product flags used as scoring inputs: NO")
print("trend_pct used: NO")
print("trend_direction used: NO")
print("Client/content IDs used as predictive features: NO")
print("IDs used only for grouping and identifying output rows: YES")
print("Baseline inputs are observed March 2026 performance fields only.")

LEAKAGE CHECK
Source window: March 2026 only
Future-window fields used: NO
FlyRank product flags used as scoring inputs: NO
trend_pct used: NO
trend_direction used: NO
Client/content IDs used as predictive features: NO
IDs used only for grouping and identifying output rows: YES
Baseline inputs are observed March 2026 performance fields only.


### Leakage-check result

The baseline uses March 2026 observed performance data only.

No future-window data, FlyRank product flags, `trend_pct`,
`trend_direction`, or other label-derived fields are used as scoring inputs.

Client and content IDs are retained only to identify and group pages.
The generated `score`, `reason_code`, and `action` are outputs of the
baseline rather than predictive inputs.

The baseline is therefore suitable as a transparent benchmark for the
later ML model, subject to the weaknesses identified in the top-20 review.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.